# Fase 0 — Setup do ambiente

Objetivo: confirmar Unity Catalog, criar catálogo/schema/volume, e deixar o `dataset.json` acessível para o Spark.

In [0]:
metastore = spark.sql("SELECT current_metastore()").collect()[0][0]
catalogo = spark.sql("SELECT current_catalog()").collect()[0][0]

print(f"Metastore atual: {metastore}")
print(f"Catálogo atual: {catalogo}")

## 1. Criar catálogo, schema e volume

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS fauna;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS fauna.monitoramento;

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS fauna.monitoramento.raw;

## Por que essa estrutura de catálogo?

O Unity Catalog organiza tudo em três níveis: **catálogo → schema → objeto**
(tabela ou volume). A decisão de nomenclatura aqui não é arbitrária — segue o
motivo de cada nível existir:

- **Catálogo `fauna`**: o catálogo é o nível de isolamento mais alto — normalmente
  representa um projeto, domínio de dados ou até uma equipe inteira. Em vez de
  jogar tudo no catálogo padrão `workspace` (que tende a virar uma gaveta bagunçada
  com o tempo, misturando projetos sem relação nenhuma), criamos um catálogo só
  nosso. Isso facilita, por exemplo, dar permissão de acesso a esse projeto
  inteiro pra alguém sem expor o resto do workspace.

- **Schema `monitoramento`**: dentro de um catálogo pode haver vários domínios de
  dados diferentes (ex.: `fauna.monitoramento`, `fauna.financeiro`, se um dia esse
  catálogo crescesse). O schema agrupa os objetos de um mesmo domínio — aqui,
  tudo relacionado ao monitoramento de fauna: as tabelas Bronze/Silver/Gold e o
  volume de dado bruto moram todos dentro de `fauna.monitoramento`.

- **Volume `raw`**: um Volume não é uma tabela — é um espaço de **arquivos**
  dentro do Unity Catalog (parecido com uma pasta, mas com governança e permissões
  como qualquer outro objeto do catálogo). Usamos ele especificamente para guardar
  o `dataset.json` bruto, porque esse arquivo ainda não é dado estruturado em
  formato de tabela — é só o arquivo de origem. As tabelas (Bronze, Silver, Gold)
  vão viver nesse mesmo schema, mas como tabelas Delta, não como arquivos soltos.

**Resumindo a organização final:**

| Objeto | Tipo | Caminho completo |
|---|---|---|
| Arquivo bruto | Volume | `fauna.monitoramento.raw` (arquivo: `dataset.json`) |
| Camada Bronze | Tabela Delta | `fauna.monitoramento.bronze_registros` |
| Camada Silver | Tabela Delta | `fauna.monitoramento.silver_registros` |
| Camada Gold | Tabela(s) Delta | `fauna.monitoramento.gold_*` |

Todo objeto sempre referenciado pelo caminho completo `catálogo.schema.objeto` —
isso evita ambiguidade quando várias pessoas ou projetos compartilham o mesmo
workspace.

## 2. Confirmar o dataset.json no volume

In [0]:
display(dbutils.fs.ls("/Volumes/fauna/monitoramento/raw/"))